<a href="https://colab.research.google.com/github/hli2325/rush-sales-analysis/blob/eda-analysis/RUSH_Sales_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RUSH Sales Analysis

<img src="https://github.com/hli2325/rush-sales-analysis/blob/main/RUSH%20logo.png?raw=true" width="200">

## 1. Project Overview
This notebook conducts an Exploratory Data Analysis (EDA) on raw sales data for RUSH, a global sportswear and footwear brand. The objective is to clean the raw data, analyze sales trends, and provide data-driven insights for company leadership.

## 2. EDA (4 Steps):
- Problem Definition
- Dataset Design & Creation
- Data Inspection
- Data Cleaning

## 3. Answer VP's Questions
- Evaluate product, state, and retailer performance.

## 4. Provide Trend Insights
- Analyze YoY growth, sales channels, and retail dynamics.

## 1 Problem Definition
**Answer VP Business Questions:**
  1. Product category with highest revenue in 2021.
  2. Top state by revenue for women's products in 2021.
  3. Top state by revenue for men's products in 2021.
  4. Top retailer by units purchased in 2020 vs. 2021.

**Trend & Insight Discovery:**

Identify key business trends across seasonality, geographical regions, and retail partners to support market growth decisions.

## 2 Data Design & Creation

### 2.1 Import Libraries & Data

- Import core libraries for data analysis and visualization.
- Import raw data from github and load into dataframes


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns

### **2.2 Product Table**

#### Dictionary

| Field | Type | Description | Note |
|---|---|---|---|
| `PRODUCT_ID` | CHAR | Unique identifier for each product type | PRIMARY KEY |
| `PRODUCT_NAME` | VARCHAR | Long product name | |

#### Notes
- The file is pipe `|`  delimited.
- text  stored as int64: `PRODUCT_ID`.

In [ ]:
# Load and preview product table
url_product = 'https://raw.githubusercontent.com/hli2325/rush-sales-analysis/refs/heads/main/TABLE_PRODUCTS_885.csv'
df_prodt = pd.read_csv(url_product, delimiter='|')
df_prodt.info()
df_prodt.head()

### **2.3 Retailer Table**

#### Dictionary

| Field | Type | Description | Note |
|---|---|---|---|
| `RETAILER_ID` | CHAR | Unique identifier for retailer-location combination | PRIMARY KEY |
| `RETAILER` | CHAR | Retailer name | |
| `REGION` | CHAR | Region of retail location | |
| `STATE` | CHAR | State of retail location | |
| `CITY` | CHAR | City of retail location | |

#### Notes:
- File data type aligns with the dictionary

In [ ]:
# Load and preview retailer table
url_retailler = 'https://github.com/hli2325/rush-sales-analysis/raw/refs/heads/main/TABLE_RETAILER_885.csv'
df_retlr = pd.read_csv(url_retailler)
df_retlr.info()
df_retlr.head()

### **2.4 Sales Table**

#### Dictionary

| Field | Type | Description | Note |
|-|-|-|-|
| `ORDER_ID` | CHAR | Unique identifier for each order | PRIMARY KEY |
| `RETAILER_ID` | CHAR | Unique identifier for retailer-location combination | FOREIGN KEY |
| `INVOICE_DATE` | DATE | Date of order | |
| `PRODUCT_ID` | CHAR | Unique identifier for each product type | FOREIGN KEY |
| `PRICE_PER_UNIT` | FLOAT | Dollar price charged per unit for the order | |
| `UNITS_SOLD` | INT | Number of units sold in the order | |
| `OPERATING_MARGIN` | FLOAT | Profit margin rate for the order | |
| `SALES_METHOD` | VARCHAR | Method of sale made ("In-store", "Outlet", or "Online") | |

#### Notes:
- text stored as int64: `ORDER_ID`, `PRODUCT_ID`.
- date stored as text: `INVOICE_DATE`.
- int stored as text: `UNIT_SOLD`.

In [ ]:
# Load and preview sales table
url_sales = 'https://github.com/hli2325/rush-sales-analysis/raw/refs/heads/main/TABLE_SALES_885.csv'
df_sales = pd.read_csv(url_sales)
df_sales.info()
df_sales.head()

## 3 Data Inspection

### 3.1 Retailer Inspection

In [ ]:
# Check missing values
print("Retailer Missing Values:\n")
print(df_retlr.isnull().sum())

# Check duplicates
print("\nRetailer Duplicate Values:\n", df_retlr.duplicated().sum())

# Check unique counts per column
print("\nRetailer Unique Values:\n")
df_retlr.nunique()

### 3.2 Sales Inspection


In [ ]:
# Check missing values
print("Sales Missing Values:")
print(df_sales.isnull().sum())

# Check duplicates
print("\nSales Duplicates Values:\n", df_sales.duplicated().sum())

# Check statistics
print("\nSales Descriptive Stats:")
display(df_sales.describe())

# Check SALES_METHOD categories
display(df_sales['SALES_METHOD'].value_counts())


In [ ]:
# Inspect PRICE_PER_UNIT outlier & missing values
display(df_sales[df_sales['PRICE_PER_UNIT'].isna() | (df_sales['PRICE_PER_UNIT'] == df_sales['PRICE_PER_UNIT'].max())])


## 4 Data Cleaning

### Identified Issues:
1. `UNITS_SOLD`: Contains corrupt `***` text strings and wrong Dtype `object`.
2. `PRICE_PER_UNIT`: Contains `99999.0` outlier and 2 missing `NaN` values.
3. `INVOICE_DATE`: Stored as text string instead of `datetime`.
4. `SALES_METHOD`: Typo `Ootlet` should be `Outlet`
5. `TOTAL_SALES`: Add calculated total revenue column.
6. Integration: Join `df_prodt`, `df_retlr`, `df_sales` into one master table.


### 4.1 Clean `UNITS_SOLD`
Remove corrupt `'***'` text rows and convert `UNITS_SOLD` Dtype to integer `int64`.


In [ ]:
# Drop 2 corrupt '***' rows and convert to int
df_sales = df_sales[df_sales['UNITS_SOLD'] != '***']
df_sales['UNITS_SOLD'] = df_sales['UNITS_SOLD'].astype(int)

### 4.2 Clean `PRICE_PER_UNIT`

- Rename `PRICE_PER_UNIT` to `UNIT_PRICE` for simplicity.
- Remove invalid `$99,999.0` outlier and missing `NaN` price rows.

In [ ]:
# Rename column name
df_sales = df_sales.rename(columns={'PRICE_PER_UNIT': 'UNIT_PRICE'})

# Drop invalid and missing values
df_sales = df_sales[(df_sales['UNIT_PRICE'] != 99999.0) & df_sales['UNIT_PRICE'].notna()]

### 4.3 Clean `INVOICE_DATE`
Convert date string values to `datetime64`.

In [ ]:
# Convert INVOICE_DATE string to datetime
df_sales['INVOICE_DATE'] = pd.to_datetime(df_sales['INVOICE_DATE'])

### 4.4 Fix `SALES_METHOD`
Replace typo `Ootlet` with `Outlet`.

In [ ]:
# Replace 'Ootlet' with 'Outlet'
df_sales['SALES_METHOD'] = df_sales['SALES_METHOD'].replace('Ootlet', 'Outlet')

### 4.5 Calculate Total Sales Revenue
Create `TOTAL_SALES` column by multiplying `UNITS_SOLD` and `UNIT_PRICE`.

In [ ]:
# Calculate Total Sales Revenue
df_sales['TOTAL_SALES'] = df_sales['UNITS_SOLD'] * df_sales['UNIT_PRICE']

### 4.6 Clean Duplicates
Remove duplicate `RETAILER_ID` entries in `df_retlr`.

In [ ]:
# Deduplicate df_retlr on RETAILER_ID
df_retlr = df_retlr.drop_duplicates(subset=['RETAILER_ID'])

### 4.7 Convert ID

Convert `ORDER_ID`, `PRODUCT_ID` from `int64` to `object`


In [ ]:
# Convert ORDER_ID, PRODUCT_ID to text
df_prodt['PRODUCT_ID'] = df_prodt['PRODUCT_ID'].astype(str)
df_sales['ORDER_ID'] = df_sales['ORDER_ID'].astype(str)
df_sales['PRODUCT_ID'] = df_sales['PRODUCT_ID'].astype(str)


### 4.8 Merge Datasets

Merge `df_sales`, `df_prodt`, `df_retlr` into `df_master`.

In [ ]:
# Merge sales with products and retailers
df_master = df_sales.merge(df_prodt, on='PRODUCT_ID', how='left')
df_master = df_master.merge(df_retlr, on='RETAILER_ID', how='left')

### 4.9 Preview Master Table

In [ ]:
# Preview Master Table
print("Null Values:\n", df_master.isnull().sum())
df_master.info()
df_master.head()

## 5 Business Questions

### 5.1 Top Sales Product Category in 2021
**Question:** What product category had the highest sales (in dollars) in 2021? How much did it sell?

In [ ]:
# Total sales by product category in 2021
q1 = df_master[df_master['YEAR'] == 2021].groupby('PRODUCT_NAME')['TOTAL_SALES'].sum().sort_values(ascending=False)
q1
# Men's Street Footwear	22649400.0


In [ ]:
# Visualize 2021 Product Sales
ax = sns.barplot(x=q1.values, y=q1.index)
ax.bar_label(ax.containers[0], fmt='${:,.0f}')



### 5.2 Top State for Women's Products (2021)
**Question:** What state had the highest sales (in dollars) of women's products in 2021? How much was it?


In [ ]:
# Filter for 2021 Women's products
q2 = df_master[(df_master['YEAR'] == 2021) & (df_master['PRODUCT_NAME'].str.contains("Women's"))]

# Total sales by state
q2 = q2.groupby('STATE')['TOTAL_SALES'].sum()

# Sort highest to lowest
q2 = q2.sort_values(ascending=False).head()
q2
# Maine	2176301.0

In [ ]:
# Visualize Top 5 States for 2021 Women's Products
ax = sns.barplot(x=q2.index, y=q2.values)
ax.bar_label(ax.containers[0], fmt='${:,.0f}')


### 5.3 Top State for Men's Products (2021)
**Question:** What state had the highest sales (in dollars) of men's products in 2021? How much was it?

In [ ]:
# Filter for 2021 Men's products
q3 = df_master[(df_master['YEAR'] == 2021) & (df_master['PRODUCT_NAME'].str.startswith("Men's"))]

# Total sales by state
q3 = q3.groupby('STATE')['TOTAL_SALES'].sum()

# Sort highest to lowest
q3 = q3.sort_values(ascending=False).head()
q3
# Delaware	2334300.0

In [ ]:
# Visualize Top 5 States for 2021 Men's Products
ax = sns.barplot(x=q3.index, y=q3.values)
ax.bar_label(ax.containers[0], fmt='${:,.0f}')


### 5.4 Top Retailer by Units Purchased (2020 vs 2021)
**Question:** What retailer purchased the most units in 2021? In 2020?

In [ ]:
# Pivot RETAILER and YEAR, sum values UNITS_SOLD
q4 = df_master.pivot_table(index='RETAILER', columns='YEAR', values='UNITS_SOLD', aggfunc='sum')

# Sort by 2021 sales
display(q4.sort_values(by=2021, ascending=False))
# Foot Locker

# Sort by 2020 sales
q4.sort_values(by=2020, ascending=False)
# Amazon

In [ ]:
# Visualize retailer units sold (2020 vs 2021)
ax = q4.plot(kind='bar',color=['blue', 'green'])
ax.margins(y=0.3)
for c in ax.containers:
    ax.bar_label(c, fmt='{:,.0f}', rotation=90, padding=3)


## 6 Additional Insights
Identify Year-over-Year (YoY) revenue growth and performance across sales channels.

### 6.1 YOY Revenue

In [ ]:
# Year-over-Year Total Revenue
yoy = df_master.groupby('YEAR')['TOTAL_SALES'].sum()
yoy

In [ ]:
# Visualize YoY Revenue Growth
ax = yoy.plot(kind='bar', color=['blue', 'green'])
ax.margins(y=0.1)
for c in ax.containers:
    ax.bar_label(c, fmt='${:,.0f}', padding=3)


### 6.2 Top Sales Method

In [ ]:
# Total Revenue by Sales Method
sales_method = df_master.groupby('SALES_METHOD')['TOTAL_SALES'].sum().sort_values(ascending=False)
ax = sns.barplot(x=sales_method.index, y=sales_method.values)
ax.bar_label(ax.containers[0], fmt='${:,.0f}')


## 7 Executive Summary

### 1. Answers to VP's Questions:
- **2021 Top Product:** Men's Street Footwear (\$22,649,400).
- **2021 Top State for Women's Products:** Maine (\$2,176,301).
- **2021 Top State for Men's Products:** Delaware (\$2,334,300).
- **2020 vs 2021 Top Retailer by Units:**
  - 2020: Amazon (316,880 units)
  - 2021: Foot Locker (1,096,890 units).

### 2. Key Growth Trends:
- **Financial Growth:** Total sales grew +296%, from \$24.17M (2020) to \$95.89M (2021).
- **Sales Channels:** Online (\$44.93M) and Outlet (\$39.53M) outperform In-store (\$35.60M).
- **Retail Partners:** Foot Locker took #1 in 2021 (1.09M units), surpassing 2020 leader Amazon (316.8K units).
